In [127]:
import pandas as pd
import numpy as np
import tensorflow as tf

In [128]:
import kagglehub
import librosa
import numpy as np
import os
from sklearn.model_selection import train_test_split
import keras
from keras.layers import Flatten, Dense, LSTM, RNN
from keras.models import Sequential
from os import listdir
from sklearn.metrics import accuracy_score
from audiomentations import Compose, AddGaussianNoise, PitchShift, HighPassFilter
from sklearn.preprocessing import StandardScaler

In [25]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

Path to dataset files: /home/pavel/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [24]:
dataset = pd.read_csv(
    '/home/pavel/.cache/kagglehub/datasets/andradaolteanu' + 
    '/gtzan-dataset-music-genre-classification/versions/1/Data/features_30_sec.csv')

In [7]:
dataset

,filename,length,chroma_stft_mean,chroma_stft_var,rms_mean,rms_var,spectral_centroid_mean,spectral_centroid_var,spectral_bandwidth_mean,spectral_bandwidth_var,...,mfcc16_var,mfcc17_mean,mfcc17_var,mfcc18_mean,mfcc18_var,mfcc19_mean,mfcc19_var,mfcc20_mean,mfcc20_var,label
0,blues.00000.wav,661794,0.350088,0.088757,0.130228,0.002827,1784.165850,129774.064525,2002.449060,85882.761315,...,52.420910,-1.690215,36.524071,-0.408979,41.597103,-2.303523,55.062923,1.221291,46.936035,blues
1,blues.00001.wav,661794,0.340914,0.094980,0.095948,0.002373,1530.176679,375850.073649,2039.036516,213843.755497,...,55.356403,-0.731125,60.314529,0.295073,48.120598,-0.283518,51.106190,0.531217,45.786282,blues
2,blues.00002.wav,661794,0.363637,0.085275,0.175570,0.002746,1552.811865,156467.643368,1747.702312,76254.192257,...,40.598766,-7.729093,47.639427,-1.816407,52.382141,-3.439720,46.639660,-2.231258,30.573025,blues
3,blues.00003.wav,661794,0.404785,0.093999,0.141093,0.006346,1070.106615,184355.942417,1596.412872,166441.494769,...,44.427753,-3.319597,50.206673,0.636965,37.319130,-0.619121,37.259739,-3.407448,31.949339,blues
4,blues.00004.wav,661794,0.308526,0.087841,0.091529,0.002303,1835.004266,343399.939274,1748.172116,88445.209036,...,86.099236,-5.454034,75.269707,-0.916874,53.613918,-4.404827,62.910812,-11.703234,55.195160,blues
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,rock.00095.wav,661794,0.352063,0.080487,0.079486,0.000345,2008.149458,282174.689224,2106.541053,88609.749506,...,45.050526,-13.289984,41.754955,2.484145,36.778877,-6.713265,54.866825,-1.193787,49.950665,rock
996,rock.00096.wav,661794,0.398687,0.075086,0.076458,0.000588,2006.843354,182114.709510,2068.942009,82426.016726,...,33.851742,-10.848309,39.395096,1.881229,32.010040,-7.461491,39.196327,-2.795338,31.773624,rock
997,rock.00097.wav,661794,0.432142,0.075268,0.081651,0.000322,2077.526598,231657.968040,1927.293153,74717.124394,...,33.597008,-12.845291,36.367264,3.440978,36.001110,-12.588070,42.502201,-2.106337,29.865515,rock
998,rock.00098.wav,661794,0.362485,0.091506,0.083860,0.001211,1398.699344,240318.731073,1818.450280,109090.207161,...,46.324894,-4.416050,43.583942,1.556207,34.331261,-5.041897,47.227180,-3.590644,41.299088,rock


In [26]:
dir1 = os.listdir(path)
path1 = os.path.join(path, dir1[0])
dir2 = os.listdir(path1)
full_path = os.path.join(path1,dir2[0])
nlabel = os.listdir(full_path)

In [133]:
def extract_audio_to_log_mel(сlass_name, patch):

    target = []
    audio = []

    for idx, name in enumerate(сlass_name):
        audio_dir = os.path.join(patch, name)
        for file in os.listdir(audio_dir):
            if file.endswith('.wav'):
                audio_file = os.path.join(audio_dir, file)
                y, sr = librosa.load(audio_file, sr = 22050, mono = True, duration = 5)
                S_mel = librosa.feature.melspectrogram(y = y, sr = sr, n_mels = 128, hop_length=512)
                s_mel_db = librosa.power_to_db(S_mel, ref=np.max)
                target.append(idx)
                audio.append(s_mel_db)
                    

    return audio, target

In [134]:
audio_mel_scpectohram, label = extract_audio_to_log_mel(nlabel, full_path)

In [135]:
X = np.array(audio_mel_scpectohram)
y = np.array(label)

In [143]:
feature_train, feature_test, label_train, label_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle = True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle = True)


In [146]:
X_train = np.expand_dims(X_train, axis = 0)
X_train = X_train.transpose(1, 2,3, 0)
X_test = np.expand_dims(X_test, axis = 0)
X_test = X_test.transpose(1, 2,3, 0)

In [147]:
X_train.shape

(798, 128, 216, 1)

In [19]:
accuracy_score_list = []

In [21]:
class lstm_keras:

    def __init__(self,X_train, X_test, y_train, y_test):
        
        self.X_train = X_train.transpose(0, 2, 1)
        self.X_test = X_test.transpose(0, 2, 1)
        self.y_train = y_train
        self.y_test = y_test
        self.model = None
        self.stopping = None
        self.predict_model = None
        self.verbose = None
        
    def stopping_model(self,verbose:int, patience:int, check:str):
        
        self.verbose = verbose
        
        model_stopping = keras.callbacks.EarlyStopping(
            monitor= check,
            patience=patience,
            mode = 'auto',
            restore_best_weights=True,
            verbose = self.verbose
            )

        self.stopping = model_stopping

    def build_model(self, lr:float):
        
        lstm_model = keras.Sequential(
            [
        keras.layers.Input(shape = (self.X_train.shape[1], self.X_train.shape[2])),
        keras.layers.BatchNormalization(),
        keras.layers.LSTM(units=64, return_sequences=True),
        keras.layers.BatchNormalization(),
        keras.layers.LSTM(units=32, kernel_regularizer= 'l1'),        
        keras.layers.Dense(units=64, activation="relu"),
        keras.layers.Dense(units = 32, kernel_regularizer= 'l2'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(units=10, activation="softmax")
            ]
                                     )
        
        optimizer = keras.optimizers.Adam(lr)
        
        lstm_model.compile(
        optimizer=optimizer,
        loss= 'sparse_categorical_crossentropy',
        metrics=["accuracy"])
        
        self.model = lstm_model
        

    def compile_model(self,n_batch:int,  n_epoch:int):
        
        history = self.model.fit(self.X_train, self.y_train,
                                      batch_size = n_batch, epochs= n_epoch,
                                      callbacks = [self.stopping], verbose = self.verbose)

    def model_predict(self):

        self.predict_model = self.model.predict(self.X_test)
        
    def scores(self):
        accuracy = accuracy_score(self.y_test, np.argmax(self.predict_model, axis = 1))
        scores = ['Имя модели: LSTM keras. Точность: ', accuracy]
        return scores

In [22]:
lstm_keras_model = lstm_keras(feature_train, feature_test, label_train, label_test)
lstm_keras_model.stopping_model(verbose = 0, patience = 10, check = 'accuracy')
lstm_keras_model.build_model(lr = 0.001)
lstm_keras_model.compile_model(n_batch = 64, n_epoch = 120)
lstm_keras_model.model_predict()
score_lstm = lstm_keras_model.scores()
accuracy_score_list.append(score_lstm)
score_lstm

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step


['Имя модели: LSTM keras. Точность: ', 0.508]

In [23]:
class GRU_keras:

    def __init__(self,X_train, X_test, y_train, y_test):
        
        self.X_train = X_train.transpose(0, 2, 1)
        self.X_test = X_test.transpose(0, 2, 1)
        self.y_train = y_train
        self.y_test = y_test
        self.model = None
        self.stopping = None
        self.predict_model = None
        self.verbose = None
        
    def stopping_model(self,verbose:int, patience:int, check:str):
        
        self.verbose = verbose
        
        model_stopping = keras.callbacks.EarlyStopping(
            monitor= check,
            patience=patience,
            mode = 'auto',
            restore_best_weights=True,
            verbose = self.verbose
            )

        self.stopping = model_stopping

    def build_model(self, lr:float):
        
        lstm_model = keras.Sequential([
        keras.layers.Input(shape = (self.X_train.shape[1], self.X_train.shape[2])),
        keras.layers.BatchNormalization(),
        keras.layers.GRU(units=64, return_sequences=True),
        keras.layers.BatchNormalization(),
        keras.layers.GRU(units=32, kernel_regularizer= 'l1'),        
        keras.layers.Dense(units=64, activation="relu"),
        keras.layers.Dense(units = 32, kernel_regularizer= 'l2'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(units=10, activation="softmax")
            ])
        
        optimizer = keras.optimizers.Adam(lr)
        
        lstm_model.compile(
        optimizer=optimizer,
        loss= 'sparse_categorical_crossentropy',
        metrics=["accuracy"])
        
        self.model = lstm_model
        

    def compile_model(self,n_batch:int,  n_epoch:int):
        
        history = self.model.fit(self.X_train, self.y_train,
                                      batch_size = n_batch, epochs= n_epoch,
                                      callbacks = [self.stopping], verbose = self.verbose)

    def model_predict(self):

        self.predict_model = self.model.predict(self.X_test)
        
    def scores(self):
        accuracy = accuracy_score(self.y_test, np.argmax(self.predict_model, axis = 1))
        scores = ['Имя модели: LSTM keras. Точность: ', accuracy]
        return scores

In [24]:
gru_keras_model = GRU_keras(feature_train, feature_test, label_train, label_test)
gru_keras_model.stopping_model(verbose = 0, patience = 10, check = 'loss')
gru_keras_model.build_model(lr = 0.01)
gru_keras_model.compile_model(n_batch = 64, n_epoch = 120)
gru_keras_model.model_predict()
score_gru = gru_keras_model.scores()
accuracy_score_list.append(score_gru)
score_gru

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 92ms/step


['Имя модели: LSTM keras. Точность: ', 0.46]

In [80]:
test_shape = np.expand_dims(audio_mel_scpectohram, axis = 0)
test_shape = test_shape.transpose(1, 2,3, 0)

In [140]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle = True)


In [141]:
X_train.shape

(798, 128, 216)

In [98]:
class cnn_keras:

    def __init__(self,X_train, X_test, y_train, y_test):
        
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.model = None
        self.stopping = None
        self.predict_model = None
        self.verbose = None
        
    def stopping_model(self,verbose:int, patience:int, check:str):
        
        self.verbose = verbose
        
        model_stopping = keras.callbacks.EarlyStopping(
            monitor= check,
            patience=patience,
            mode = 'auto',
            restore_best_weights=True,
            verbose = self.verbose
            )

        self.stopping = model_stopping

    def build_model(self, lr:float):
        
        cnn_model = keras.Sequential([
            keras.layers.Input(shape=(self.X_train.shape[1], self.X_train.shape[2], self.X_train.shape[3])),
            keras.layers.BatchNormalization(),
            keras.layers.Conv2D(2, (2,2), activation='relu', padding='same' , kernel_regularizer= 'l1'),
            keras.layers.Dropout(0.2),
            keras.layers.MaxPooling2D((2,2)),
            keras.layers.Dropout(0.2),
            keras.layers.Flatten(),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(32), 
            keras.layers.Dense(10, activation='softmax')
        ])
        
        optimizer = keras.optimizers.Adam(lr)
        
        cnn_model.compile(
        optimizer=optimizer,
        loss= 'sparse_categorical_crossentropy',
        metrics=["accuracy"])
        
        self.model = cnn_model
        

    def compile_model(self,n_batch:int,  n_epoch:int):
        
        history = self.model.fit(self.X_train, self.y_train,
                                      batch_size = n_batch, epochs= n_epoch,
                                      callbacks = [self.stopping], verbose = self.verbose)

    def model_predict(self):

        self.predict_model = self.model.predict(self.X_test)
        
    def scores(self):
        accuracy = accuracy_score(self.y_test, np.argmax(self.predict_model, axis = 1))
        scores = ['Имя модели: CNN keras. Точность: ', accuracy]
        return scores

In [148]:
cnn_model = cnn_keras(X_train, X_test, y_train, y_test)
cnn_model.stopping_model(verbose = 0, patience = 10, check = 'loss')
cnn_model.build_model(lr = 0.01)
cnn_model.compile_model(n_batch = 32, n_epoch = 30)
cnn_model.model_predict()
cnn_model_score = cnn_model.scores()
cnn_model_score

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


['Имя модели: CNN keras. Точность: ', 0.45]

In [161]:
class cnn_and_rnn_keras:

    def __init__(self,X_train, X_test, y_train, y_test):
        
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.model = None
        self.stopping = None
        self.predict_model = None
        self.verbose = None
        
    def stopping_model(self,verbose:int, patience:int, check:str):
        
        self.verbose = verbose
        
        model_stopping = keras.callbacks.EarlyStopping(
            monitor= check,
            patience=patience,
            mode = 'auto',
            restore_best_weights=True,
            verbose = self.verbose
            )

        self.stopping = model_stopping

    def build_model(self, lr:float):

        
        inputs = layers.Input(shape=(self.X_train.shape[1], self.X_train.shape[2], self.X_train.shape[3]))
        x = layers.BatchNormalization()(inputs)
        x = layers.Conv2D(2, (2,2), activation='relu', padding='same', kernel_regularizer='l1')(x)
        x = layers.Dropout(0.2)(x)
        x = layers.MaxPooling2D((2,2))(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Permute((2, 1, 3))(x)          
        
       
        times = x.shape[1]
        features = x.shape[2] * x.shape[3]
        x = layers.Reshape((times, features))(x)
        
        x = layers.LSTM(64)(x)
        x = layers.Dense(32)(x)
        outputs = layers.Dense(10, activation='softmax')(x)

        self.model = keras.Model(inputs, outputs)
        
        optimizer = keras.optimizers.Adam(lr)
        
        self.model.compile(
        optimizer=optimizer,
        loss= 'sparse_categorical_crossentropy',
        metrics=["accuracy"])
        
        

    def compile_model(self,n_batch:int,  n_epoch:int):
        
        history = self.model.fit(self.X_train, self.y_train,
                                      batch_size = n_batch, epochs= n_epoch,
                                      callbacks = [self.stopping], verbose = self.verbose)

    def model_predict(self):

        self.predict_model = self.model.predict(self.X_test)
        
    def scores(self):
        accuracy = self.model.evaluate(self.X_train, self.y_train)
        scores = ['Имя модели: CNN + LSTM keras. Точность: ', accuracy]
        return scores

In [162]:
cnn_model_and_lstm = cnn_and_rnn_keras(X_train, X_test, y_train, y_test)
cnn_model_and_lstm.stopping_model(verbose = 0, patience = 10, check = 'loss')
cnn_model_and_lstm.build_model(lr = 0.01)
cnn_model_and_lstm.compile_model(n_batch = 32, n_epoch = 30)
cnn_model_and_lstm.model_predict()
cnn_model_and_lstm_score = cnn_model_and_lstm.scores()
cnn_model_and_lstm_score

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9825 - loss: 0.1036


['Имя модели: CNN + LSTM keras. Точность: ',
 [0.1036028116941452, 0.9824561476707458]]

In [ ]:
cnn ,rnn ,rnn + attention ,cnn + rnn 